In [ ]:
import torch
from torch import nn
from typing import Optional, Tuple


In [ ]:
class KVCacheManager:
    def __init__(self, n_layers, n_heads, max_seq_len, dim_head, device):
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.dim_head = dim_head
        self.device = device

        # preallocate KV cache for all layers
        self.K = [torch.zeros(n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.V = [torch.zeros(n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.curr_len = 0  # tokens stored

    def update(self, layer_id, K_new, V_new):
        t = self.curr_len
        self.K[layer_id][:, t, :] = K_new.squeeze(0)
        self.V[layer_id][:, t, :] = V_new.squeeze(0)
        if layer_id == self.n_layers - 1:
            self.curr_len += 1

    def get(self, layer_id):
        return self.K[layer_id][:, :self.curr_len, :], self.V[layer_id][:, :self.curr_len, :]

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        B, T, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # reshape for heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            K_all, V_all = kv_cache.get(layer_id)
            K_all = torch.cat([K_all, k], dim=1)
            V_all = torch.cat([V_all, v], dim=1)
        else:
            K_all, V_all = k, v

        # attention computation
        attn = (q @ K_all.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = attn @ V_all

        # merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out), k, v

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim, n_heads)
        self.mlp = MLP(dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        # Attention block
        attn_out, K_new, V_new = self.attn(self.ln1(x), kv_cache, layer_id)
        x = x + attn_out
        # MLP block
        x = x + self.mlp(self.ln2(x))
        return x, K_new, V_new

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids, kv_cache=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for layer_id, block in enumerate(self.blocks):
            x, K_new, V_new = block(x, kv_cache, layer_id)
            if kv_cache is not None:
                kv_cache.update(layer_id, K_new, V_new)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [ ]:
def sample_top_k(logits, k=50, temperature=1.0):
    logits = logits / temperature
    top_k_vals, top_k_idx = torch.topk(logits, k)
    probs = torch.softmax(top_k_vals, dim=-1)
    idx = torch.multinomial(probs, 1)
    return top_k_idx.gather(-1, idx)

In [ ]:
def generate(model, input_ids, kv_cache, max_new_tokens):
    generated = input_ids.clone()
    for step in range(max_new_tokens):
        # last token only
        x = generated[:, -1:]
        logits = model(x, kv_cache)
        next_token = sample_top_k(logits[:, -1, :])
        generated = torch.cat([generated, next_token.unsqueeze(-1)], dim=1)
    return generated